In [ ]:
run_date = None
config_path = 'config/settings.yaml'


In [ ]:
%matplotlib inline
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import yaml
from IPython.display import display

from qis_risk_report.data.loaders import (
    common_date_range,
    load_portfolio,
    load_returns,
    load_weights,
)
from qis_risk_report.reports import charts
from qis_risk_report.risk import metrics as m
from qis_risk_report.risk.portfolio import (
    build_portfolio_contribution_grid,
    marginal_var,  # noqa: F401
    )
from qis_risk_report.risk.scenarios import (
    ShockParams,
    default_scenarios,
    replay_scenario,
    run_scenario,
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=0.9)

cfg = yaml.safe_load(Path(config_path).read_text(encoding='utf-8-sig'))
qis_return_df = load_returns(cfg['data']['qis_returns_path'])
portfolio_return_df = load_portfolio(cfg['data']['portfolio_returns_path'])
weights_df = load_weights(cfg['data']['weights_path'])

weights = weights_df.iloc[-1]
sub_cols = [c for c in qis_return_df.columns if c != 'total']
port_cols = list(portfolio_return_df.columns)
port_weights = weights.reindex(port_cols).fillna(0)

_overlap_start, _overlap_end = common_date_range(qis_return_df, portfolio_return_df)
_hyp_weights = cfg.get('hypothetical_qis_weights', [0.01, 0.025, 0.05, 0.10])

_rf = float(cfg.get('risk_free_rate', 0.0))
_run_date = run_date or qis_return_df.index[-1].strftime('%Y-%m-%d')
_run_dt = pd.Timestamp(_run_date)
_ytd_start = _run_dt.replace(month=1, day=1)
_mtd_start = _run_dt.replace(day=1)

print(
    f'Report date: {_run_date}  |  '
    f'QIS: {qis_return_df.index[0].date()} to {qis_return_df.index[-1].date()}  |  '
    f'{len(qis_return_df)} trading days'
)
print(f'Portfolio overlap: {_overlap_start.date()} to {_overlap_end.date()}')


# QIS Risk Report

---


## Summary


In [ ]:
_s = qis_return_df['total']
_s252 = _s.iloc[-252:]

_ytd_s = _s.loc[_ytd_start:_run_date]
_ytd_ret = float((1 + _ytd_s).prod() - 1) if len(_ytd_s) > 0 else float('nan')
_1y_ann = m.annualised_return(_s252) if len(_s252) >= 2 else float('nan')
_ann_vol_1y = m.annualised_volatility(_s252)
_sharpe_1y = (_1y_ann - _rf) / _ann_vol_1y if _ann_vol_1y > 0 else float('nan')
_max_dd = m.max_drawdown(_s)
_curr_dd = float(m.drawdown_series(_s).iloc[-1])
_var99_1d = m.historical_var(_s252, confidence=0.99)

kpi = pd.DataFrame({
    'KPI': [
        'YTD Return',
        '1Y Annualised Return',
        'Annualised Volatility (1Y)',
        'Sharpe Ratio (1Y)',
        'Maximum Drawdown',
        'Current Drawdown',
        '1-Day VaR 99%',
    ],
    'Value': [
        f'{_ytd_ret:.2%}',
        f'{_1y_ann:.2%}',
        f'{_ann_vol_1y:.2%}',
        f'{_sharpe_1y:.2f}',
        f'{_max_dd:.2%}',
        f'{_curr_dd:.2%}',
        f'{_var99_1d:.3%}',
    ],
}).set_index('KPI')
display(kpi)


---

## 1. Recent Performance


In [ ]:
def _period_return(s, start, end):
    w = s.loc[start:end]
    return float((1 + w).prod() - 1) if len(w) > 0 else float('nan')

rows = []
for col in qis_return_df.columns:
    s = qis_return_df[col]
    _1y_data = s.iloc[-252:]
    rows.append({
        'Component': col,
        '1D': f"{float(s.iloc[-1]):.2%}",
        'MTD': f"{_period_return(s, _mtd_start, _run_date):.2%}",
        'YTD': f"{_period_return(s, _ytd_start, _run_date):.2%}",
        '1Y': f"{float((1 + _1y_data).prod() - 1):.2%}",
        '1Y Ann.': f"{m.annualised_return(_1y_data):.2%}" if len(_1y_data) >= 2 else 'N/A',
        'Inception': f"{float((1 + s).prod() - 1):.2%}",
    })
display(pd.DataFrame(rows).set_index('Component'))


In [ ]:
fig = charts.plot_daily_return_bars(qis_return_df, window=63)
plt.tight_layout()
plt.show()


In [ ]:
fig = charts.plot_rolling_sharpe(qis_return_df, window=252, risk_free=_rf)
plt.tight_layout()
plt.show()

---

## 2. Risk Metrics


In [ ]:
rows = []
for col in qis_return_df.columns:
    s = qis_return_df[col]
    s252 = s.iloc[-252:]
    rows.append({
        'Component': col,
        'Ann Vol 252d': f'{m.annualised_volatility(s252):.2%}',
        'Ann Vol All': f'{m.annualised_volatility(s):.2%}',
        'Sharpe 252d': f'{m.sharpe_ratio(s252, _rf):.2f}',
        'Sharpe All': f'{m.sharpe_ratio(s, _rf):.2f}',
        'Sortino 252d': f'{m.sortino_ratio(s252, _rf):.2f}',
        'Sortino All': f'{m.sortino_ratio(s, _rf):.2f}',
        'Max DD': f'{m.max_drawdown(s):.2%}',
        'Curr DD': f'{float(m.drawdown_series(s).iloc[-1]):.2%}',
        'Max DD Dur (d)': m.drawdown_duration(s),
        'VaR 95% (252d)': f'{m.historical_var(s252, 0.95):.3%}',
        'VaR 99% (252d)': f'{m.historical_var(s252, 0.99):.3%}',
        '10d VaR 99%': f'{m.historical_var(s252, 0.99, 10):.3%}',
        'CVaR 95% (252d)': f'{m.expected_shortfall(s252, 0.95):.3%}',
    })
display(pd.DataFrame(rows).set_index('Component'))


In [ ]:
fig = charts.plot_rolling_volatility(qis_return_df)
plt.tight_layout()
plt.show()


In [ ]:
fig = charts.plot_underwater(qis_return_df)
plt.tight_layout()
plt.show()


In [ ]:
fig = charts.plot_return_distribution(qis_return_df, confidence=0.95)
plt.tight_layout()
plt.show()


In [ ]:
fig = charts.plot_correlation_heatmap(qis_return_df)
plt.tight_layout()
plt.show()


In [ ]:
fig = charts.plot_rolling_pairwise_correlation(qis_return_df, window=252)
plt.tight_layout()
plt.show()


---

## 3. Scenario Analysis


In [ ]:
_all_scenarios = default_scenarios()

rows = []
for scen in _all_scenarios:
    try:
        result = replay_scenario(scen, qis_return_df)
        window = qis_return_df.loc[scen.start:scen.end]
        has_data = 'total' in window.columns and len(window) >= 2
        vol = m.annualised_volatility(window['total']) if has_data else float('nan')
        row = {
            'Event': scen.name,
            'Start': scen.start,
            'End': scen.end,
            'Days': result.metadata.get('n_days', 0),
            'Strategy P&L': f'{result.pnl_total:.2%}',
        }
        for k, v in result.pnl_by_component.items():
            row[k] = f'{v:.2%}'
        row['Ann. Vol'] = f'{vol:.2%}'
        rows.append(row)
    except Exception as exc:
        rows.append({
            'Event': scen.name, 'Start': scen.start, 'End': scen.end,
            'Days': 0, 'Strategy P&L': f'N/A ({exc})',
        })

_scenario_results = []
for scen in _all_scenarios:
    try:
        _scenario_results.append(replay_scenario(scen, qis_return_df))
    except Exception:
        pass

display(pd.DataFrame(rows).set_index('Event'))


In [ ]:
fig = charts.plot_scenario_impact(_scenario_results)
plt.tight_layout()
plt.show()


In [ ]:
fig = charts.plot_stress_path('GFC', qis_return_df, _all_scenarios)
plt.tight_layout()
plt.show()


In [ ]:
fig = charts.plot_stress_path('COVID Crash', qis_return_df, _all_scenarios)
plt.tight_layout()
plt.show()


In [ ]:
_spot_shocks = cfg.get('synthetic_shocks', {}).get('spot_shocks', [-0.05, -0.025, 0.0, 0.025, 0.05])
_vol_shocks = cfg.get('synthetic_shocks', {}).get('vol_shocks', [-0.20, 0.0, 0.20])

shock_grid = {}
for spot in _spot_shocks:
    row_data = {}
    for vol in _vol_shocks:
        result = run_scenario(qis_return_df[sub_cols], ShockParams(spot_shock=spot, vol_shock=vol))
        row_data[f'vol {vol:+.0%}'] = f'{result.pnl_total:.2%}'
    shock_grid[f'spot {spot:+.1%}'] = row_data

display(pd.DataFrame(shock_grid).T)


---

## 4. Portfolio Risk Contribution


In [ ]:
_grid = build_portfolio_contribution_grid(
    qis_return_df, portfolio_return_df, weights_df, _hyp_weights, confidence=0.95
)

_grid_display = _grid.copy()
_grid_display['standalone_var'] = _grid_display['standalone_var'].map('{:.3%}'.format)
_grid_display['marginal_var'] = _grid_display['marginal_var'].map('{:.4%}'.format)
_grid_display['component_var_share'] = (
    _grid_display['component_var_share'].map('{:.1%}'.format)
)
_grid_display['correlation'] = _grid_display['correlation'].map('{:.3f}'.format)
_grid_display['diversification_benefit'] = (
    _grid_display['diversification_benefit'].map('{:.4%}'.format)
)
_grid_display.index.name = 'QIS Weight'
_grid_display.columns = [
    'Standalone VaR 95%', 'Marginal VaR', 'Component VaR Share',
    'Correlation', 'Diversification Benefit',
]
display(_grid_display)


In [ ]:
fig = charts.plot_allocation_sensitivity(_grid)
plt.tight_layout()
plt.show()


In [ ]:
_qis_overlap = qis_return_df.loc[_overlap_start:_overlap_end]
_port_overlap = portfolio_return_df.loc[_overlap_start:_overlap_end]
fig = charts.plot_rolling_qis_portfolio_correlation(
    _qis_overlap, _port_overlap, port_weights, window=252
)
plt.tight_layout()
plt.show()


In [ ]:
fig = charts.plot_cumulative_return(qis_return_df)
plt.tight_layout()
plt.show()


In [ ]:
fig = charts.plot_monthly_heatmap(qis_return_df)
plt.tight_layout()
plt.show()
